# SC-NeuroCore Neuron Explorer

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anulum/sc-neurocore/blob/main/notebooks/04_neuron_explorer.ipynb)

Explore **117 neuron models** from the `models` subpackage (plus 5 core SC neurons at the package level = 122 total) spanning 82 years of computational neuroscience (McCulloch-Pitts 1943 → ArcaneNeuron 2026).

Pick any model, drive it with current, and see voltage traces, spike rasters, phase portraits, and firing rate curves — all in one place.

In [ ]:
# !pip install -q sc-neurocore matplotlib numpy

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sc_neurocore.neurons import models

ALL_MODELS = sorted(models.__all__)
print(f"{len(ALL_MODELS)} neuron models available")
print("Families: LIF variants, Hodgkin-Huxley, Izhikevich, adaptive,")
print("  oscillatory, bursting, map-based, multi-compartment, hardware")
print("  chip emulators, population, rate, AI-optimized, and more.")

## 1. Single-Model Deep Dive

Change `MODEL_NAME` below to explore any of the 117 models.

In [ ]:
MODEL_NAME = "HodgkinHuxleyNeuron"  # Change this to any model name
DURATION = 500    # timesteps
I_DRIVE = 10.0    # input current amplitude

neuron = getattr(models, MODEL_NAME)()
print(f"Model: {MODEL_NAME}")
print(f"Parameters: {vars(neuron) if hasattr(neuron, '__dict__') else 'dataclass'}")

In [ ]:
# Drive the neuron and record everything
voltages, spikes, currents = [], [], []
for t in range(DURATION):
    I = I_DRIVE * (1.0 + 0.3 * np.sin(2 * np.pi * t / 100))
    currents.append(I)
    result = neuron.step(I)
    spike = int(result) if not isinstance(result, (tuple, list)) else int(result[0])
    spikes.append(spike)
    v = getattr(neuron, 'v', getattr(neuron, 'voltage', getattr(neuron, 'vs', 0.0)))
    voltages.append(float(v) if not isinstance(v, (list, np.ndarray)) else float(v[0]) if len(v) > 0 else 0.0)

spike_times = np.where(spikes)[0]
print(f"Spikes: {len(spike_times)} / {DURATION} steps ({len(spike_times)/DURATION:.1%} firing rate)")

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True,
                          gridspec_kw={'height_ratios': [3, 1, 1]})

# Voltage trace
axes[0].plot(voltages, linewidth=0.8, color='#2c3e50')
for st in spike_times:
    axes[0].axvline(st, color='#e74c3c', alpha=0.4, linewidth=0.5)
axes[0].set_ylabel('Membrane Potential')
axes[0].set_title(f'{MODEL_NAME} \u2014 Voltage Trace')

# Spike raster
if len(spike_times) > 0:
    axes[1].eventplot([spike_times], colors='#e74c3c', linewidths=1.5)
axes[1].set_ylabel('Spikes')
axes[1].set_yticks([])

# Input current
axes[2].plot(currents, linewidth=0.8, color='#27ae60')
axes[2].set_ylabel('Input Current')
axes[2].set_xlabel('Time Step')

plt.tight_layout()
plt.show()

In [ ]:
# Phase portrait (voltage vs derivative)
if len(voltages) > 1:
    dv = np.diff(voltages)
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.plot(voltages[:-1], dv, linewidth=0.3, color='#2c3e50', alpha=0.6)
    ax.scatter(voltages[0], dv[0], color='#27ae60', s=60, zorder=5, label='Start')
    ax.set_xlabel('V(t)')
    ax.set_ylabel('dV/dt')
    ax.set_title(f'{MODEL_NAME} \u2014 Phase Portrait')
    ax.legend()
    plt.tight_layout()
    plt.show()

## 2. Model Family Comparison

Compare firing patterns across different neuron families under identical input.

In [ ]:
COMPARE = [
    ("LapicqueNeuron", 5.0),
    ("ExpIFNeuron", 5.0),
    ("HodgkinHuxleyNeuron", 10.0),
    ("FitzHughNagumoNeuron", 1.0),
    ("HindmarshRoseNeuron", 3.0),
    ("AdExNeuron", 5.0),
    ("MorrisLecarNeuron", 100.0),
    ("ArcaneNeuron", 5.0),
]
T = 300

fig, axes = plt.subplots(len(COMPARE), 1, figsize=(12, 2 * len(COMPARE)), sharex=True)

for idx, (name, amp) in enumerate(COMPARE):
    n = getattr(models, name)()
    vs = []
    for t in range(T):
        I = amp * (1.0 + 0.3 * np.sin(2 * np.pi * t / 80))
        n.step(I)
        v = getattr(n, 'v', getattr(n, 'voltage', getattr(n, 'vs', getattr(n, 'v_fast', 0.0))))
        vs.append(float(v) if not isinstance(v, (list, np.ndarray)) else float(v[0]) if hasattr(v, '__len__') and len(v) > 0 else 0.0)
    axes[idx].plot(vs, linewidth=0.8)
    axes[idx].set_ylabel(name.replace('Neuron', ''), fontsize=8)
    axes[idx].tick_params(labelsize=7)

axes[-1].set_xlabel('Time Step')
fig.suptitle('Firing Pattern Comparison \u2014 8 Models, Same Sinusoidal Input', fontsize=13)
plt.tight_layout()
plt.show()

## 3. F-I Curve (Firing Rate vs Input Current)

Sweep input current from 0 to `I_MAX` and measure the steady-state firing rate.

In [ ]:
FI_MODEL = "LapicqueNeuron"
I_MAX = 15.0
N_STEPS = 30
SIM_LEN = 500

currents_sweep = np.linspace(0, I_MAX, N_STEPS)
rates = []
for I in currents_sweep:
    n = getattr(models, FI_MODEL)()
    spk = sum(int(n.step(I)) for _ in range(SIM_LEN))
    rates.append(spk / SIM_LEN)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(currents_sweep, rates, 'o-', markersize=4, color='#2c3e50')
ax.set_xlabel('Input Current')
ax.set_ylabel('Firing Rate')
ax.set_title(f'{FI_MODEL} \u2014 F-I Curve')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Stochastic Computing Pipeline

SC-NeuroCore's core capability: encode analog values as bitstreams, process with logic gates, decode back.

In [ ]:
from sc_neurocore import BitstreamEncoder, generate_bernoulli_bitstream, bitstream_to_probability
from sc_neurocore import RNG

# Encode two probabilities as bitstreams
p_a, p_b = 0.7, 0.4
length = 2048

bs_a = generate_bernoulli_bitstream(p_a, length, rng=RNG(seed=42))
bs_b = generate_bernoulli_bitstream(p_b, length, rng=RNG(seed=99))

# AND gate = multiplication in SC
bs_product = bs_a & bs_b
p_result = bitstream_to_probability(bs_product)

print(f"Input: {p_a} x {p_b} = {p_a * p_b:.4f} (exact)")
print(f"SC:    popcount(A AND B) / {length} = {p_result:.4f}")
print(f"Error: {abs(p_result - p_a * p_b):.4f}")

# Visualise the first 64 bits
fig, axes = plt.subplots(3, 1, figsize=(12, 3))
for i, (bs, label) in enumerate([(bs_a, f'A (p={p_a})'), (bs_b, f'B (p={p_b})'), (bs_product, f'A AND B (p\u2248{p_a*p_b:.2f})')]):
    axes[i].imshow(bs[:64].reshape(1, -1), aspect='auto', cmap='Greys', interpolation='none')
    axes[i].set_ylabel(label, fontsize=8)
    axes[i].set_yticks([])
axes[-1].set_xlabel('Bit position')
fig.suptitle('Stochastic Computing: AND gate = Multiplication', fontsize=12)
plt.tight_layout()
plt.show()

## 5. Full Model Catalogue

Every model available, sorted alphabetically.

In [ ]:
for i, name in enumerate(ALL_MODELS, 1):
    print(f"{i:3d}. {name}")

## Next Steps

- **Rust engine**: build the optional bridge from the repo checkout for 512x speedup
- **HDC/VSA**: `from sc_neurocore_engine import HDCVector` for 10,000-bit hyperdimensional computing
- **FPGA export**: `from sc_neurocore.ir import SCCompiler` to emit synthesisable SystemVerilog
- **Training**: `from sc_neurocore.training import LIFCell` for surrogate gradient training (99.49% MNIST)
- **Analysis**: 126 spike train analysis functions in `sc_neurocore.analysis`
- **Full docs**: [anulum.github.io/sc-neurocore](https://anulum.github.io/sc-neurocore/)